# `makimoto-kawa` 0.2.0 showcase

What's new in this release, demoed live against **production** (intentional, see Setup below):

- **API key auth** (was a dashboard JWT before, now a static `api_key`)
- **Pagination** on `list_jobs()`
- **Filters**: `status`, `job_type`, `language`, `created_after`, `job_id`
- **`iter_jobs()`**: auto-paginating convenience over the above
- **`create_summary()` / `create_tags()`**: new job types derived from a completed transcription
- **`Job.type` / `Job.source_job_id` / `Job.audio_seconds`**: new/recovered fields

**Renamed in this release**, if you've seen an earlier version of this SDK: `list_transcriptions` → `list_jobs`, `get_transcription` → `get_job`, `delete_transcription` → `delete_job`, `iter_transcriptions` → `iter_jobs`, one shared set of methods now covers all three job types (transcription, summary, tags), not just transcriptions.

## Setup


In [ ]:
import os

from makimoto.kawa import KawaClient

# default api_url = production, on purpose
client = KawaClient(api_key=os.environ["MAKIMOTO_API_KEY"])

## 1. Get one real, completed transcription

`create_summary()`/`create_tags()` each accept **either** `transcription_job_id` (one of your own transcriptions, in status `succeeded`) **or** `transcript_text` (raw text, no transcription job involved at all), exactly one of the two, the API rejects zero or both with a 400.

Section 6 below shows both paths, once each: summary from this transcription, tags from raw text directly.

In [ ]:
AUDIO = "audio/harvard.wav"

transcription = client.transcribe(AUDIO)
print(transcription.job_id, transcription.type, transcription.status)

## 2. Pagination

`list_jobs()` returns a `TranscriptionPage`: `.transcriptions` (this page's jobs) and `.next_cursor` (`None` once there's nothing left).

In [ ]:
page = client.list_jobs(limit=7)
print(f"{len(page.transcriptions)} jobs on this page, cursor = {page.next_cursor!r}")
for job in page.transcriptions:
    print(" ", job.job_id, job.type, job.status)

In [ ]:
if page.next_cursor:
    next_page = client.list_jobs(limit=7, cursor=page.next_cursor)
    n = len(next_page.transcriptions)
    print(f"page 2: {n} jobs, cursor = {next_page.next_cursor!r}")
    for job in next_page.transcriptions:
        print(" ", job.job_id, job.type, job.status)
else:
    print("no next page — list_jobs(limit=2) already returned everything")

## 4. Filters, including the new `job_type`

`status`, `job_type` (`"transcription"`, `"summary"`, or `"tags"`), `language`, `created_after`, `job_id`, composed with AND when combined. With no `job_type`, every job type comes back.

In [ ]:
summary_only = client.list_jobs(job_type="summary")
print("job_type=summary:", [j.job_id for j in summary_only.transcriptions])

## 5. `iter_jobs()`: auto-paginating convenience

Same filters, but walks every page automatically, a generator, so it's lazy and stops early if you do too (e.g. via `itertools.islice`).

In [ ]:
import itertools

matches = client.iter_jobs(job_type="transcription", page_size=3)
for job in itertools.islice(matches, 100):
    print(job.job_id, job.status, job.language)

## 6. Summary and tags: one from a job, one from raw text

`create_summary()`/`create_tags()` both take either path. Below: summary from the transcription above (`transcription_job_id`), tags from raw text directly (`transcript_text`), one demo of each path is enough, the other method would work identically either way.

In [ ]:
summary_job = client.create_summary(
    transcript_text="Customer called about a billing issue on their March invoice, "
    "agent applied a one-time credit and confirmed the next cycle would reflect it."
)
final = summary_job
for update in client.poll(summary_job.job_id, interval=2):
    final = update

print("type:", final.type, " source_job_id:", final.source_job_id)
if final.status == "succeeded":
    print("topic:", final.result.topic)
    print("summary:", final.result.summary)
else:
    print("didn't succeed:", final.error)

In [ ]:
tags_job = client.create_tags(transcription.job_id)
final = tags_job
for update in client.poll(tags_job.job_id, interval=2):
    final = update

# source_job_id: ties this tags job back to the transcription above
print("type:", final.type, " source_job_id:", final.source_job_id)
if final.status == "succeeded":
    print("tags:", final.result.tags)
else:
    print("didn't succeed:", final.error)

## 7. Everything in one list, by type

The transcription and the summary job derived from it both show up in the same `list_jobs()` call, `job_type` tells them apart, `source_job_id` ties the summary back to the transcription it came from (the tags job has none, it came from raw text, no transcription behind it).

In [ ]:
everything = client.list_jobs(job_id=transcription.job_id)
audio_seconds = everything.transcriptions[0].audio_seconds
print("original transcription, audio_seconds:", audio_seconds)

for job in [client.get_job(summary_job.job_id), client.get_job(tags_job.job_id)]:
    print(job.job_id, job.type, "derived from", job.source_job_id)